# Congress trades around committee work and legislation

Pull House members' trades in a sector or stock, then line them up against bills and hearings you track, with trade and report dates.

Read it with charts at https://www.quantisnow.com/sec-api/guides/congress-trades-before-legislation. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

Members report trades up to 45 days late, so the question is usually historical: who traded defense stocks before an appropriations vote, or a drug maker before a hearing. The API gives each trade with both dates and the member's state and party. The bill and hearing calendar comes from you.

## House members' trades in one stock

### Which House members traded Nvidia this year?

Everyone who bought or sold one stock, with the dollar range each reported.

```shell
curl -X POST https://www.quantisnow.com/api/v1/congress/trades \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "ticker:NVDA AND tradedOn:[2026-01-01 TO 2026-11-24]",
  "from": 0,
  "size": 5,
  "sort": [{"tradedOn": {"order": "desc"}}]
}'
```

Response (real, first 2 of 45):

```json
{
  "total": {
    "value": 45,
    "relation": "eq"
  },
  "data": [
    {
      "id": "20036855-6",
      "reportId": "20036855",
      "chamber": "house",
      "member": {
        "name": "Michael A. Rulli",
        "bioguideId": "R000619",
        "state": "OH",
        "district": "OH06",
        "party": "R"
      },
      "owner": "self",
      "ticker": "NVDA",
      "asset": "NVIDIA Corporation - Common Stock (NVDA)",
      "assetType": "ST",
      "type": "partial_sale",
      "tradedOn": "2026-10-28",
      "notifiedOn": "2026-10-28",
      "filedOn": "2026-11-18",
      "amount": {
        "min": 1001,
        "max": 15000
      },
      "description": "Subholding Of: Merrill Lynch SIMPLE Managed Account",
      "reportUrl": "https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2026/20036855.pdf"
    },
    {
      "id": "20035778-3",
      "reportId": "20035778",
      "chamber": "house",
      "member": {
        "name": "Maria Elvira Salazar",
        "bioguideId": "S000168",
        "state": "FL",
        "district": "FL27",
        "party": "R"
      },
      "owner": "self",
      "ticker": "NVDA",
      "asset": "NVIDIA Corporation - Common Stock (NVDA)",
      "assetType": "ST",
      "type": "purchase",
      "tradedOn": "2026-10-23",
      "notifiedOn": "2026-11-01",
      "filedOn": "2026-11-05",
      "amount": {
        "min": 1001,
        "max": 15000
      },
      "description": "Filing Status: New; Subholding Of: UBS Brokerage Account",
      "reportUrl": "https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2026/20035778.pdf"
    }
  ]
}
```

## Trades in a window before a date, in Python

In [ ]:
import os, requests
from datetime import date, timedelta

EVENT = date(2026, 9, 30)          # a vote, hearing or bill introduction you track
TICKERS = ["LMT", "RTX", "NOC", "GD"]
start, end = EVENT - timedelta(days=60), EVENT

res = requests.post("https://www.quantisnow.com/api/v1/congress/trades",
    headers={"Authorization": f"Bearer {os.environ['QUANTISNOW_API_KEY']}"},
    json={"query": f"ticker:({' OR '.join(TICKERS)}) AND tradedOn:[{start} TO {end}]",
          "sort": "tradedOn:asc", "size": 50}).json()
for t in res["data"]:
    m = t["member"]
    print(t["tradedOn"], m["name"], f'{m["party"]}-{m["state"]}', t["type"], t["ticker"], t["amount"]["min"], t["amount"]["max"])

## From the command line

```shell
npx quantisnow congress NVDA
npx quantisnow query congress/trades 'ticker:(LMT OR RTX) AND tradedOn:[2026-08-01 TO 2026-09-30]'
```

A trade before an event isn't evidence of anything by itself. Members trade often, spouses and managed accounts trade for them (`owner` says whose account), and amounts are ranges. Use it to find what to read, not to conclude.

Related: [Congress trading API](https://www.quantisnow.com/sec-api/guides/congress-stock-trading-api).